Load Climate Dataset

In [ ]:

from pathlib import Path
import torch

# 1. Project directory
project_root = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "experiments" / "02_climate" / "data" / "wind_nearest_5sites.pt").is_file()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Climate dataset checkpoint not found")

experiment_dir = project_root / "experiments" / "02_climate"
dataset_path = experiment_dir / "data" / "wind_nearest_5sites.pt"

# 2. Load dataset
dataset = torch.load(dataset_path, map_location="cpu")

x_in = dataset["x_in"]
x_out = dataset["x_out"]

patch_len = dataset["patch_len"]
horizon = dataset["horizon"]
input_dim = x_in.shape[-1]

# 3. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== CLIMATE EXPERIMENT ===")
print("Device:", device)
print("X:", x_in.shape)
print("Y:", x_out.shape)
print("Input dimension:", input_dim)
print("Patch length:", patch_len)
print("Forecast horizon:", horizon)


=== CLIMATE EXPERIMENT ===
Device: cuda
X: torch.Size([310, 80, 5])
Y: torch.Size([310, 50])
Input dimension: 5
Patch length: 80
Forecast horizon: 10


Strict Stable Koopman Operator

In [2]:
import torch.nn as nn

def _orth(w: torch.Tensor) -> torch.Tensor:
    return torch.linalg.qr(w)[0]


class StrictStableKoopmanOperator(nn.Module):
    def __init__(self, latent_dim: int, ρ_max: float = 0.99):
        super().__init__()
        self.U_raw = nn.Parameter(torch.randn(latent_dim, latent_dim))
        self.V_raw = nn.Parameter(torch.randn(latent_dim, latent_dim))
        self.S_raw = nn.Parameter(torch.randn(latent_dim))
        self.ρ_max = ρ_max

    def forward(self, z: torch.Tensor):
        U, V = _orth(self.U_raw), _orth(self.V_raw)
        Σ = torch.sigmoid(self.S_raw) * self.ρ_max
        K = U @ torch.diag(Σ) @ V.T
        return z @ K.T, K, Σ


Positional Encoding, Patch Embedding and Series Decomposition

In [3]:
import numpy as np

# 1. Positional Encoding
class SinCosPosEnc(nn.Module):
    def __init__(self, d_model: int, max_len: int = 10_000):
        super().__init__()
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10_000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.size(1)]


# 2. Patch Embedding
class PatchEmbed1D(nn.Module):
    def __init__(self, in_ch: int, d_model: int, patch_len: int, stride: int):
        super().__init__()
        self.conv = nn.Conv1d(in_ch, d_model, kernel_size=patch_len, stride=stride)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.conv(x)
        return x.permute(0, 2, 1)


# 3. Series Decomposition
class SeriesDecomp(nn.Module):
    def __init__(self, k: int = 3):
        super().__init__()
        self.avg = nn.AvgPool1d(k, stride=1, padding=k // 2)

    def forward(self, x):
        trend = self.avg(x.transpose(1, 2)).transpose(1, 2)
        if trend.size(1) != x.size(1):
            trend = trend[:, :x.size(1)]
        return trend, x - trend


PatchTST Backbone and Koopformer

In [4]:

class PatchTST_Backbone(nn.Module):
    def __init__(self, input_dim: int, seq_len: int, patch_len: int, d_model: int = 48, num_layers: int = 3, num_heads: int = 4, dim_ff: int = 96):
        super().__init__()
        self.patch = PatchEmbed1D(input_dim, d_model, patch_len, patch_len)
        n_patches = int(np.ceil(seq_len / patch_len))
        self.pos = SinCosPosEnc(d_model, max_len=n_patches + 1)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

    def forward(self, x):
        x = self.patch(x)
        cls = self.cls.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.encoder(self.pos(x))
        return x[:, 0]


class Koopformer_PatchTST(nn.Module):
    def __init__(self, input_dim, seq_len, horizon, patch_len, d_model=48):
        super().__init__()
        self.backbone = PatchTST_Backbone(input_dim, seq_len, patch_len=patch_len, d_model=d_model)
        self.koop = StrictStableKoopmanOperator(d_model)
        self.fc = nn.Linear(d_model, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred


Autoformer Backbone and Koopformer

In [5]:

class SimpleAutoformer(nn.Module):
    def __init__(self, input_len, horizon, input_dim, patch_len, d_model=48, num_heads=4, dim_ff=64, num_layers=3):
        super().__init__()
        self.dec = SeriesDecomp(k=patch_len)
        self.embed = nn.Linear(input_dim, d_model)
        self.pos = SinCosPosEnc(d_model, max_len=input_len)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)
        self.fc_seas = nn.Linear(d_model, horizon)
        self.fc_trend = nn.Linear(input_len * input_dim, horizon)

    def forward(self, x):
        trend, seas = self.dec(x)
        seas = self.encoder(self.pos(self.embed(seas)))
        seas_o = self.fc_seas(seas.mean(1))
        trend_o = self.fc_trend(trend.reshape(trend.size(0), -1))
        return seas_o + trend_o


class Koopformer_Autoformer(nn.Module):
    def __init__(self, input_dim, seq_len, horizon,
                 patch_len, d_model=16):
        super().__init__()
        self.backbone = SimpleAutoformer(seq_len, horizon, input_dim, patch_len=patch_len, d_model=d_model)
        self.koop = StrictStableKoopmanOperator(horizon)
        self.fc = nn.Linear(horizon, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred


Informer Backbone and Koopformer

In [7]:

class InformerSparse(nn.Module):
    def __init__(self, input_dim, d_model=48, num_heads=4, dim_ff=96, num_layers=3, seq_len=120, patch_len=1):
        super().__init__()
        self.use_patch = patch_len > 1
        if self.use_patch:
            self.patch = PatchEmbed1D(input_dim, d_model, patch_len, patch_len)
            n_tokens = int(np.ceil(seq_len / patch_len))
        else:
            self.embed = nn.Linear(input_dim, d_model)
            n_tokens = seq_len
        self.pos = SinCosPosEnc(d_model, max_len=n_tokens)
        enc = nn.TransformerEncoderLayer(d_model, num_heads, dim_ff, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)

    def forward(self, x):
        if self.use_patch:
            x = self.patch(x)
        else:
            x = self.embed(x)
        x = self.encoder(self.pos(x))
        return x.mean(dim=1)


class Koopformer_Informer(nn.Module):
    def __init__(self, input_dim, seq_len, horizon, patch_len, d_model=48):
        super().__init__()
        self.backbone = InformerSparse(input_dim, d_model=d_model, seq_len=seq_len, patch_len=patch_len)
        self.koop = StrictStableKoopmanOperator(d_model)
        self.fc = nn.Linear(d_model, horizon * input_dim)

    def forward(self, x, return_latents=False):
        z = self.backbone(x)
        z_next, _, _ = self.koop(z)
        pred = self.fc(z_next)
        if return_latents:
            return pred, z, z_next
        return pred

LSTM Baseline

In [ ]:
class SimpleLSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, horizon):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, horizon * input_dim)

    def forward(self, x, return_latents=False):
        _, (h_n, _) = self.lstm(x)
        z = h_n[-1]
        out = self.fc(z)
        if return_latents:
            return out, z, z
        return out